# Hitter Deep Dive Analysis

**Purpose:** Advanced analysis of hitter WAR predictions

**Prerequisites:** Must run hitter_pipeline_main.ipynb first

**Last Updated:** 2025-10-06

---

## Analysis Overview
1. Elite hitter performance (>5 WAR/600)
2. Position-specific performance analysis
3. Enhanced feature impact (Baserunning, Defense, Positional_WAR)
4. Positional adjustment validation
5. Feature correlation heatmap
6. Partial dependence plots (AVG, OBP, SLG)
7. SHAP values for top 20 hitters
8. Error analysis by year and team
9. Prediction interval analysis
10. Model component comparison
11. Outlier investigation (>2 sigma residuals)

In [ ]:
# Cell 1: Imports and Load Saved Data

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

# Load most recent predictions
predictions_dir = project_root / 'predictions'
pred_files = [f for f in os.listdir(predictions_dir) if 'hitter' in f and f.endswith('.csv')]
latest_pred_file = sorted(pred_files)[-1]
df_predictions = pd.read_csv(predictions_dir / latest_pred_file)

print(f"Loaded predictions: {latest_pred_file}")
print(f"  Total predictions: {len(df_predictions)}")
if 'Primary_Position' in df_predictions.columns:
    print(f"  Positions: {df_predictions['Primary_Position'].value_counts().to_dict()}")

# Load saved model
models_dir = project_root / 'models'
model_files = [f for f in os.listdir(models_dir) if 'hitter' in f and f.endswith('.pkl')]
latest_model_file = sorted(model_files)[-1]
hitter_model = joblib.load(models_dir / latest_model_file)

print(f"\nLoaded model: {latest_model_file}")
print("Ready for deep-dive analysis!")

In [ ]:
# Cell 2: Elite Hitter Analysis (>5 WAR/600)

# Calculate residuals if not already in predictions
if 'Residual' not in df_predictions.columns and 'Actual_WAR_per_600' in df_predictions.columns:
    df_predictions['Residual'] = df_predictions['Actual_WAR_per_600'] - df_predictions['Predicted_WAR_per_600']

# Filter elite hitters
elite_threshold = 5.0

if 'Actual_WAR_per_600' in df_predictions.columns:
    elite_hitters = df_predictions[df_predictions['Actual_WAR_per_600'] > elite_threshold]
    
    print("=" * 60)
    print(f"ELITE HITTERS (>{elite_threshold} WAR/600)")
    print("=" * 60)
    print(f"Count: {len(elite_hitters)}")
    print(f"Mean Actual WAR: {elite_hitters['Actual_WAR_per_600'].mean():.2f}")
    print(f"Mean Predicted WAR: {elite_hitters['Predicted_WAR_per_600'].mean():.2f}")
    print(f"Mean Residual: {elite_hitters['Residual'].mean():.2f}")
    print(f"MAE: {elite_hitters['Residual'].abs().mean():.2f}")
    
    # Show biggest under-predictions
    print("\nTop 5 Under-predicted Elite Hitters:")
    under_predicted = elite_hitters.nlargest(5, 'Residual')
    display_cols = ['Name', 'PA', 'Actual_WAR_per_600', 'Predicted_WAR_per_600', 'Residual']
    if 'Primary_Position' in under_predicted.columns:
        display_cols.insert(1, 'Primary_Position')
    print(under_predicted[display_cols].to_string(index=False))
else:
    print("Note: Actual WAR data not available in predictions (current season projections only)")

In [ ]:
# Cell 3: Position-Specific Performance Analysis

if 'Primary_Position' in df_predictions.columns and 'Residual' in df_predictions.columns:
    # Analyze MAE by position
    position_performance = df_predictions.groupby('Primary_Position').agg({
        'Residual': ['mean', 'std', lambda x: np.abs(x).mean(), 'count']
    }).round(3)
    
    position_performance.columns = ['Mean Error', 'Std Dev', 'MAE', 'Count']
    position_performance = position_performance[position_performance['Count'] >= 10]  # Min 10 players
    
    print("=" * 60)
    print("PERFORMANCE BY POSITION")
    print("=" * 60)
    print(position_performance.to_string())
    
    # Visualization
    import plotly.express as px
    fig = px.box(df_predictions, x='Primary_Position', y='Residual',
                title="Residual Distribution by Position")
    fig.add_hline(y=0, line_dash="dash", line_color="red")
    fig.show()
    
    print("\nAnalysis:")
    print("- Which positions have highest prediction accuracy?")
    print("- Are catchers systematically under/over-predicted?")
    print("- Does positional adjustment work as expected?")
else:
    print("Position or residual data not available")

In [ ]:
# Cell 4: Enhanced Feature Impact Analysis

from new_pipeline.notebooks.shared.plotting_utils import create_partial_dependence
from new_pipeline.notebooks.shared.pipeline_runner import load_historical_data, run_data_pipeline
from new_pipeline.common.constants import HITTER_MODEL_FEATURES

# Load historical data for analysis
print("Loading historical data for enhanced feature analysis...")
hitter_historical = load_historical_data(player_type='hitter', years=[2024])
hitter_processed = run_data_pipeline(hitter_historical, player_type='hitter')

# Check which enhanced features are available
enhanced_features = ['Enhanced_Baserunning', 'Enhanced_Defense', 'Positional_WAR']
available_enhanced = [f for f in enhanced_features if f in HITTER_MODEL_FEATURES]

if available_enhanced:
    print(f"\nEnhanced features in model: {available_enhanced}")
    
    X_hitters = hitter_processed[HITTER_MODEL_FEATURES].values
    
    # Create PDPs for each enhanced feature
    for feat in available_enhanced:
        fig = create_partial_dependence(hitter_model, X_hitters, feature=feat, feature_names=HITTER_MODEL_FEATURES)
        fig.update_layout(title=f"Partial Dependence: {feat}")
        fig.show()
    
    print("\nAnalysis Questions:")
    print("- Is the relationship linear or non-linear?")
    print("- What's the marginal WAR value of +1 baserunning run?")
    print("- Does defense matter more for certain positions?")
else:
    print("Enhanced features not found in model")

In [ ]:
# Cell 5: Positional Adjustment Validation

import plotly.graph_objects as go

if 'Positional_WAR' in HITTER_MODEL_FEATURES and 'Primary_Position' in hitter_processed.columns:
    # Validate Positional_WAR adjustments make sense
    position_adj = hitter_processed.groupby('Primary_Position')['Positional_WAR'].mean().sort_values()
    
    print("=" * 60)
    print("AVERAGE POSITIONAL ADJUSTMENT BY POSITION")
    print("=" * 60)
    print(position_adj.to_string())
    
    print("\nExpected order (hardest to easiest):")
    print("  C > SS > 2B > CF > 3B > RF > LF > 1B > DH")
    
    # Visualize
    fig = go.Figure(go.Bar(x=position_adj.index, y=position_adj.values))
    fig.update_layout(title="Average Positional Adjustment",
                     xaxis_title="Position", yaxis_title="WAR Adjustment per 600 PA")
    fig.add_hline(y=0, line_dash="dash")
    fig.show()
else:
    print("Positional_WAR feature or position data not available")

In [ ]:
# Cell 6: Feature Correlation Heatmap

from new_pipeline.notebooks.shared.plotting_utils import create_correlation_heatmap

# Add WAR for correlation
feature_cols_with_war = HITTER_MODEL_FEATURES + ['WAR']
available_cols = [c for c in feature_cols_with_war if c in hitter_processed.columns]

fig = create_correlation_heatmap(hitter_processed, features=available_cols)
fig.update_layout(title="Hitter Feature Correlation Matrix (2024)")
fig.show()

print("\nAnalysis Questions:")
print("- Which features are highly correlated? (e.g., AVG and OBP)")
print("- Which features are independent predictors?")
print("- How do enhanced features correlate with traditional stats?")

In [ ]:
# Cell 7: Partial Dependence Plot - AVG

if 'AVG' in HITTER_MODEL_FEATURES:
    fig_avg = create_partial_dependence(hitter_model, X_hitters, feature='AVG', feature_names=HITTER_MODEL_FEATURES)
    fig_avg.update_layout(title="Partial Dependence: AVG")
    fig_avg.show()
    
    print("\nExpected Pattern: Positive relationship (higher AVG -> higher WAR)")
else:
    print("AVG feature not in model features")

In [ ]:
# Cell 8: Partial Dependence Plot - OBP

if 'OBP' in HITTER_MODEL_FEATURES:
    fig_obp = create_partial_dependence(hitter_model, X_hitters, feature='OBP', feature_names=HITTER_MODEL_FEATURES)
    fig_obp.update_layout(title="Partial Dependence: OBP")
    fig_obp.show()
    
    print("\nExpected Pattern: Strong positive relationship (OBP highly correlated with WAR)")
else:
    print("OBP feature not in model features")

In [ ]:
# Cell 9: Partial Dependence Plot - SLG

if 'SLG' in HITTER_MODEL_FEATURES:
    fig_slg = create_partial_dependence(hitter_model, X_hitters, feature='SLG', feature_names=HITTER_MODEL_FEATURES)
    fig_slg.update_layout(title="Partial Dependence: SLG")
    fig_slg.show()
    
    print("\nExpected Pattern: Positive relationship (power contributes to WAR)")
else:
    print("SLG feature not in model features")

In [ ]:
# Cell 10: SHAP Values - Top 20 Hitters

from new_pipeline.notebooks.shared.analysis_utils import calculate_shap_values
import shap
import matplotlib.pyplot as plt

# Calculate SHAP for top 20 hitters by WAR
if 'WAR' in hitter_processed.columns:
    top_20 = hitter_processed.nlargest(20, 'WAR')
    X_top_20 = top_20[HITTER_MODEL_FEATURES].values
    
    print("Calculating SHAP values for top 20 hitters...")
    print("This may take 2-5 minutes...\n")
    
    shap_values = calculate_shap_values(hitter_model, X_top_20, background_samples=100)
    
    # Waterfall plot for #1 hitter
    print(f"SHAP Analysis: {top_20.iloc[0]['Name']}")
    shap.plots.waterfall(shap_values[0], show=False)
    plt.title(f"SHAP Analysis: {top_20.iloc[0]['Name']}")
    plt.show()
    
    # Summary plot for all top 20
    shap.plots.beeswarm(shap_values, show=False)
    plt.title("Feature Impact on Top 20 Hitters")
    plt.show()
    
    print("\nAnalysis Questions:")
    print("- Which features drive elite predictions?")
    print("- Is defense or offense more important for top hitters?")
    print("- How much do enhanced features matter?")
else:
    print("WAR data not available for SHAP analysis")

In [ ]:
# Cell 11: Error Analysis by Year

import plotly.graph_objects as go

# Load predictions for multiple years (if available)
years_available = [2022, 2023, 2024]
error_by_year = {}

for year in years_available:
    try:
        year_files = [f for f in os.listdir(predictions_dir) 
                     if 'hitter' in f and str(year) in f and f.endswith('.csv')]
        if year_files:
            df_year = pd.read_csv(predictions_dir / year_files[0])
            if 'Residual' in df_year.columns:
                residuals = df_year['Residual'].values
                error_by_year[year] = {
                    'MAE': np.abs(residuals).mean(),
                    'RMSE': np.sqrt((residuals**2).mean()),
                    'Mean Error': residuals.mean()
                }
    except Exception as e:
        continue

if error_by_year:
    # Plot year-over-year performance
    years_list = list(error_by_year.keys())
    mae_list = [error_by_year[y]['MAE'] for y in years_list]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years_list, y=mae_list, mode='lines+markers',
                             name='MAE', line=dict(width=3)))
    fig.update_layout(title="Model Performance Over Time",
                     xaxis_title="Year", yaxis_title="MAE")
    fig.show()
    
    print("Year-by-year MAE:")
    for year in years_list:
        print(f"  {year}: {error_by_year[year]['MAE']:.3f}")
else:
    print("Multi-year prediction data not available yet")

In [ ]:
# Cell 12: Error Analysis by Team

if 'Team' in df_predictions.columns and 'Residual' in df_predictions.columns:
    # Analyze if certain teams have systematically better/worse predictions
    team_errors = df_predictions.groupby('Team')['Residual'].agg(['mean', 'std', 'count'])
    team_errors = team_errors[team_errors['count'] >= 5]  # Teams with 5+ hitters
    
    # Top 5 over-predicted teams
    print("=" * 60)
    print("TEAMS WITH LARGEST OVER-PREDICTIONS")
    print("=" * 60)
    print(team_errors.nsmallest(5, 'mean').to_string())
    
    # Top 5 under-predicted teams
    print("\n" + "=" * 60)
    print("TEAMS WITH LARGEST UNDER-PREDICTIONS")
    print("=" * 60)
    print(team_errors.nlargest(5, 'mean').to_string())
    
    print("\nNote: Could indicate park factor issues or team-specific strategies")
else:
    print("Team data or residuals not available for analysis")

In [ ]:
# Cell 13: Model Component Comparison and Outlier Investigation

from new_pipeline.notebooks.shared.analysis_utils import compare_models, find_outliers
import plotly.express as px

# Part 1: Model Component Comparison
base_models = {}
if hasattr(hitter_model, 'rf_model'):
    base_models['RandomForest'] = hitter_model.rf_model
if hasattr(hitter_model, 'keras_model'):
    base_models['Keras'] = hitter_model.keras_model
if hasattr(hitter_model, 'xgb_model'):
    base_models['XGBoost'] = hitter_model.xgb_model

if len(base_models) > 0 and 'WAR' in hitter_processed.columns:
    X_test = X_hitters
    y_test = hitter_processed['WAR_per_600'].values
    
    comparison = compare_models(base_models, X_test, y_test)
    
    print("=" * 60)
    print("MODEL COMPONENT COMPARISON")
    print("=" * 60)
    print(comparison.to_string(index=False))
    
    # Visualize
    fig = px.bar(comparison, x='Model', y=['MAE', 'RMSE'],
                title="Ensemble Component Performance",
                barmode='group')
    fig.show()

# Part 2: Outlier Investigation
print("\n" + "=" * 60)
if 'Residual' in df_predictions.columns:
    outlier_mask = find_outliers(df_predictions['Residual'].values, threshold=2.0)
    outliers = df_predictions[outlier_mask]
    
    print("OUTLIERS (>2 sigma residuals)")
    print("=" * 60)
    print(f"Count: {len(outliers)} ({len(outliers)/len(df_predictions):.1%})")
    
    # Investigate top outliers
    print("\nLargest Residuals:")
    display_cols = ['Name', 'PA']
    if 'Primary_Position' in outliers.columns:
        display_cols.append('Primary_Position')
    if 'Actual_WAR_per_600' in outliers.columns:
        display_cols.extend(['Actual_WAR_per_600', 'Predicted_WAR_per_600', 'Residual'])
    
    print(outliers.nlargest(10, 'Residual')[display_cols].to_string(index=False))
    
    # Common characteristics?
    if 'Primary_Position' in outliers.columns:
        print("\nOutlier Position Distribution:")
        print(outliers['Primary_Position'].value_counts().to_string())
    
    print("\nAnalysis Questions:")
    print("- Are outliers concentrated in certain positions?")
    print("- Are they mostly low-PA hitters (small sample noise)?")
    print("- Are there specific teams or parks?")
else:
    print("Residual data not available for outlier analysis")

print("\n" + "=" * 60)
print("HITTER DEEP DIVE COMPLETE")
print("=" * 60)